# TM016: Hybrid Transformers (Frozen Embeddings)
This notebook implements the architecture of using fine-tuned Hugging Face Transformers as frozen feature extractors for classical ML classifiers.

In [ ]:
import os
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.metrics import f1_score, classification_report
import matplotlib.pyplot as plt
import seaborn as sns

from src.evaluate import evaluate_and_log
from src.config import SEED

## 1. Load Pre-Extracted Embeddings
Run `python src/sentence_embeddings.py` beforehand to populate the `outputs/embeddings/` folder.

In [ ]:
out_dir = "../outputs/embeddings"
y_train = np.load(os.path.join(out_dir, "y_train.npy"))
y_val = np.load(os.path.join(out_dir, "y_val.npy"))

features = {
    "distilbert_cls": ("X_train_distilbert_cls.npy", "X_val_distilbert_cls.npy"),
    "distilbert_mean": ("X_train_distilbert_mean.npy", "X_val_distilbert_mean.npy"),
    "finbert_cls": ("X_train_finbert_cls.npy", "X_val_finbert_cls.npy"),
    "finbert_mean": ("X_train_finbert_mean.npy", "X_val_finbert_mean.npy"),
}

X_data = {}
for name, (tr_file, va_file) in features.items():
    try:
        X_data[name] = {
            "train": np.load(os.path.join(out_dir, tr_file)),
            "val": np.load(os.path.join(out_dir, va_file))
        }
        print(f"Loaded {name}: {X_data[name]['train'].shape}")
    except FileNotFoundError:
        print(f"Missing {name} features. Run python src/sentence_embeddings.py first.")

## 2. Train Classifiers and Compare

In [ ]:
results = []

for feat_name, data in X_data.items():
    # 1. Logistic Regression
    lr = LogisticRegression(penalty="l2", solver="lbfgs", max_iter=1000, class_weight="balanced", random_state=SEED)
    lr.fit(data["train"], y_train)
    y_pred_lr = lr.predict(data["val"])
    f1_lr = f1_score(y_val, y_pred_lr, average="macro")
    results.append({"Features": feat_name, "Model": "Logistic Regression", "F1 Macro": f1_lr})
    
    # 2. XGBoost
    xgb = XGBClassifier(max_depth=3, learning_rate=0.1, n_estimators=150, random_state=SEED, n_jobs=-1)
    xgb.fit(data["train"], y_train)
    y_pred_xgb = xgb.predict(data["val"])
    f1_xgb = f1_score(y_val, y_pred_xgb, average="macro")
    results.append({"Features": feat_name, "Model": "XGBoost", "F1 Macro": f1_xgb})

df_results = pd.DataFrame(results).sort_values("F1 Macro", ascending=False)
display(df_results)

In [ ]:
plt.figure(figsize=(10, 6))
sns.barplot(data=df_results, x="F1 Macro", y="Features", hue="Model")
plt.title("Frozen Transformer Embeddings: Feature & Model Comparison")
plt.tight_layout()
plt.show()